# KaggleResearch v2.0

**Literature-seeded, self-directed autoresearch for Kaggle competitions**

This notebook automatically:
1. Parses your Kaggle competition and creates a baseline
2. Searches academic literature for SOTA approaches
3. Selects a strategy and generates experiments
4. Runs an exploit loop, keeping winning changes
5. Re-researches when progress plateaus
6. Branches the codebase for strategy pivots

---

## Cell 0: Install Dependencies & Setup

In [ ]:
# Install required packages not pre-installed in Colab
!pip install -q kaggle anthropic arxiv segmentation_models_pytorch

# Mount Google Drive for persistence (project data will be saved here)
from google.colab import drive
drive.mount('/content/drive')

# Clone KaggleResearch from GitHub
import os
import sys

KAGGLERESEARCH_PATH = '/content/kaggleresearch'

# Clone or pull latest
if os.path.exists(KAGGLERESEARCH_PATH):
    print("Updating kaggleresearch...")
    !cd {KAGGLERESEARCH_PATH} && git pull
else:
    print("Cloning kaggleresearch...")
    !git clone https://github.com/benjaminwfriedman/kaggleresearch.git {KAGGLERESEARCH_PATH}

# Add to Python path
if KAGGLERESEARCH_PATH not in sys.path:
    sys.path.insert(0, KAGGLERESEARCH_PATH)

# Verify installation
utils_path = os.path.join(KAGGLERESEARCH_PATH, 'utils')
if os.path.exists(utils_path):
    print(f"✓ KaggleResearch installed at {KAGGLERESEARCH_PATH}")
else:
    raise FileNotFoundError("Failed to clone kaggleresearch. Check the GitHub URL.")

print("✓ Dependencies installed and Drive mounted!")

## Cell 1: Configuration

**Edit these values before running!**

In [ ]:
# ─── API CREDENTIALS ─────────────────────────────────────────────────────────
# Set these before running!
KAGGLE_API_TOKEN     = ''       # Your Kaggle API token (from kaggle.com/settings > API)
ANTHROPIC_API_KEY    = ''       # Your Anthropic API key
# ─────────────────────────────────────────────────────────────────────────────

# ─── KaggleResearch Config ───────────────────────────────────────────────────
COMPETITION_URL      = 'https://www.kaggle.com/competitions/titanic'  # Your competition URL
DRIVE_PATH           = '/content/drive/MyDrive/kaggleresearch'        # Where to store project data

# Experiment timing
TIME_BUDGET_MIN      = 4        # Minutes per experiment (3-5 for Colab)

# Plateau detection - re-research triggers when:
# fewer than PLATEAU_MIN_GAIN_PCT % improvement over last PLATEAU_WINDOW experiments
PLATEAU_WINDOW       = 5        # Number of recent experiments to evaluate
PLATEAU_MIN_GAIN_PCT = 1.0      # Minimum cumulative % gain over that window

# Branch comparison - when a pivot happens, how many experiments to run
# on the new branch before comparing against the old best
BRANCH_COMPARE_N     = 5

# Literature
LITERATURE_DEPTH     = 10       # Papers retrieved per search (5-20)
RUN_LIT_REVIEW       = True     # Set False to skip if STRATEGY.md already exists
# ─────────────────────────────────────────────────────────────────────────────

# Setup Kaggle credentials BEFORE any kaggle imports
import os
if KAGGLE_API_TOKEN:
    os.makedirs('/root/.kaggle', exist_ok=True)
    with open('/root/.kaggle/access_token', 'w') as f:
        f.write(KAGGLE_API_TOKEN)
    os.chmod('/root/.kaggle/access_token', 0o600)

# Validate config
assert COMPETITION_URL, "Please set COMPETITION_URL"
assert DRIVE_PATH, "Please set DRIVE_PATH"
assert KAGGLE_API_TOKEN, "Please set KAGGLE_API_TOKEN"
assert ANTHROPIC_API_KEY, "Please set ANTHROPIC_API_KEY"

print(f"Competition: {COMPETITION_URL}")
print(f"Drive path: {DRIVE_PATH}")
print("Kaggle API token: configured")

## Cell 2: Setup & Authentication

In [ ]:
import os
import sys
import json
import shutil
from pathlib import Path
from datetime import datetime

# Import from utils
from utils.kaggle_api import extract_slug_from_url

COMPETITION_SLUG = extract_slug_from_url(COMPETITION_URL)
PROJECT_DIR = Path(DRIVE_PATH) / COMPETITION_SLUG
REPO_DIR = PROJECT_DIR / 'repo'
DATA_DIR = REPO_DIR / 'data'
LITERATURE_DIR = PROJECT_DIR / 'literature_cache'

# Create directories
for d in [PROJECT_DIR, REPO_DIR, DATA_DIR, LITERATURE_DIR, REPO_DIR / 'submissions']:
    d.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")

# Kaggle credentials already set up in Cell 1
print("Kaggle API token: configured")

# Initialize Anthropic client
import anthropic
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("Anthropic client initialized")

# Detect GPU
from utils.experiment_runner import detect_gpu, scale_time_budget
GPU_NAME, HAS_GPU = detect_gpu()
SCALED_TIME_BUDGET = scale_time_budget(TIME_BUDGET_MIN, GPU_NAME)
print(f"\nGPU: {GPU_NAME}")
print(f"Time budget: {SCALED_TIME_BUDGET:.1f} min/experiment")

## Cell 3: Bootstrap Phase

Parse competition, download data, create baseline.

In [ ]:
from utils.kaggle_api import (
    parse_competition, classify_problem_type, 
    download_competition_data, get_template_for_problem_type
)
from utils.checkpoint import (
    load_checkpoint, save_checkpoint, detect_phase,
    create_initial_checkpoint, CheckpointState
)
from utils.branching import init_repo

CHECKPOINT_PATH = PROJECT_DIR / 'checkpoint.json'
STRATEGY_PATH = PROJECT_DIR / 'STRATEGY.md'
IDEAS_PATH = PROJECT_DIR / 'IDEAS.md'
DB_PATH = PROJECT_DIR / 'experiment_log.db'

# Check for existing checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)
current_phase = detect_phase(checkpoint)
print(f"Current phase: {current_phase}")

if current_phase == 'bootstrap' or checkpoint is None:
    print("\n=== Bootstrap Phase ===")
    
    # Parse competition
    print("Parsing competition metadata...")
    competition_meta = parse_competition(COMPETITION_URL)
    print(f"  Name: {competition_meta.name}")
    print(f"  Metric: {competition_meta.metric}")
    
    # Download data
    print("\nDownloading competition data...")
    downloaded_files = download_competition_data(COMPETITION_SLUG, DATA_DIR)
    print(f"  Downloaded {len(downloaded_files)} files")
    
    # Classify problem type
    print("\nClassifying problem type...")
    problem_type = classify_problem_type(competition_meta, DATA_DIR)
    print(f"  Problem type: {problem_type}")
    
    # Copy baseline template
    template_name = get_template_for_problem_type(problem_type)
    template_path = Path(KAGGLERESEARCH_PATH) / 'templates' / template_name
    train_path = REPO_DIR / 'train.py'
    
    if template_path.exists():
        shutil.copy(template_path, train_path)
        print(f"\n✓ Copied baseline template: {template_name}")
    else:
        print(f"\n⚠ Template not found: {template_path}")
    
    # Initialize git repo
    init_repo(REPO_DIR)
    print("✓ Initialized git repository")
    
    # Run baseline to get initial score
    print("\nRunning baseline experiment...")
    from utils.experiment_runner import run_experiment
    baseline_score, error = run_experiment(REPO_DIR, SCALED_TIME_BUDGET)
    
    if baseline_score is None:
        print(f"\n⚠ Baseline failed: {error}")
        print("Please fix train.py and re-run this cell")
        baseline_score = 0.0
    else:
        print(f"\n✓ Baseline score: {baseline_score:.6f}")
    
    # Save baseline score
    with open(REPO_DIR / 'baseline_score.txt', 'w') as f:
        f.write(str(baseline_score))
    
    # Create initial checkpoint
    checkpoint = create_initial_checkpoint(
        competition_slug=COMPETITION_SLUG,
        problem_type=problem_type,
        baseline_score=baseline_score
    )
    checkpoint.strategy_name = "Baseline"
    
    # Store competition meta in checkpoint
    checkpoint_dict = checkpoint.to_dict()
    checkpoint_dict['competition_meta'] = {
        'name': competition_meta.name,
        'metric': competition_meta.metric,
        'metric_direction': competition_meta.metric_direction,
    }
    
    save_checkpoint(CHECKPOINT_PATH, checkpoint)
    print("\n✓ Bootstrap complete!")
else:
    print(f"Resuming from checkpoint (phase: {current_phase})")
    print(f"  Best score: {checkpoint.best_score:.6f}")
    print(f"  Experiments: {checkpoint.total_experiments}")

## Cell 4: Literature Review & Strategy Selection

In [ ]:
from utils.literature import (
    search_papers, cache_papers, load_cached_papers,
    save_search_history, build_search_query, format_papers_for_prompt
)
from utils.strategy import select_strategy, generate_ideas_md

# Reload checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)

if RUN_LIT_REVIEW and (not STRATEGY_PATH.exists() or checkpoint.phase == 'literature'):
    print("=== Literature Review Phase ===")
    
    # Load competition meta from checkpoint
    checkpoint_dict = checkpoint.to_dict() if hasattr(checkpoint, 'to_dict') else vars(checkpoint)
    comp_meta = checkpoint_dict.get('competition_meta', {})
    
    # Build search query
    query = build_search_query(
        problem_type=checkpoint.problem_type,
        metric=comp_meta.get('metric', 'accuracy')
    )
    print(f"Search query: {query}")
    
    # Search for papers
    print("\nSearching academic literature...")
    papers = search_papers(
        query=query,
        n=LITERATURE_DEPTH,
        problem_type=checkpoint.problem_type
    )
    print(f"  Found {len(papers)} relevant papers")
    
    # Cache papers
    papers_cache = LITERATURE_DIR / 'papers.json'
    cache_papers(papers, papers_cache)
    save_search_history(query, LITERATURE_DIR / 'search_history.json')
    
    # Format for prompt
    papers_summary = format_papers_for_prompt(papers)
    
    # Generate strategy
    print("\nGenerating strategy...")
    strategy_content = select_strategy(
        papers_summary=papers_summary,
        competition_meta={
            'name': comp_meta.get('name', COMPETITION_SLUG),
            'problem_type': checkpoint.problem_type,
            'metric': comp_meta.get('metric', 'accuracy'),
            'metric_direction': comp_meta.get('metric_direction', 'higher_better'),
            'description': '',
            'baseline_model': 'LightGBM',
        },
        baseline_score=checkpoint.baseline_score,
        client=client
    )
    
    with open(STRATEGY_PATH, 'w') as f:
        f.write(strategy_content)
    print(f"✓ STRATEGY.md written")
    
    # Generate IDEAS.md
    print("\nGenerating experiment ideas...")
    ideas_content = generate_ideas_md(
        papers_summary=papers_summary,
        strategy_md=strategy_content,
        competition_meta={
            'name': comp_meta.get('name', COMPETITION_SLUG),
            'problem_type': checkpoint.problem_type,
        },
        baseline_score=checkpoint.baseline_score,
        client=client
    )
    
    with open(IDEAS_PATH, 'w') as f:
        f.write(ideas_content)
    print(f"✓ IDEAS.md written")
    
    # Update checkpoint
    checkpoint.phase = 'exploit'
    save_checkpoint(CHECKPOINT_PATH, checkpoint)
    
else:
    print("Skipping literature review (STRATEGY.md exists or RUN_LIT_REVIEW=False)")

# Display strategy and ideas for review
from utils.display import render_strategy_and_ideas_widget, display_in_colab
html = render_strategy_and_ideas_widget(STRATEGY_PATH, IDEAS_PATH)
display_in_colab(html)

## Cell 5: Human Checkpoint

**Review STRATEGY.md and IDEAS.md above.**

Edit them directly in Google Drive if needed, then click the button below to start the experiment loop.

In [ ]:
from utils.display import create_approval_button

APPROVED = False

def approve_and_continue():
    global APPROVED
    APPROVED = True
    print("\n" + "="*50)
    print("Strategy approved! Run the next cell to start the experiment loop.")
    print("="*50)

create_approval_button(approve_and_continue)

## Cell 6: Exploit Phase (Main Loop)

This cell runs the experiment loop. It will:
1. Pick the next idea from IDEAS.md
2. Implement it using the LLM
3. Run the experiment
4. Keep improvements, revert failures
5. Detect plateaus and trigger re-research

In [ ]:
import time
from datetime import datetime
from IPython.display import clear_output

from utils.checkpoint import (
    load_checkpoint, save_checkpoint,
    update_checkpoint_after_experiment,
    update_checkpoint_for_reresearch
)
from utils.strategy import (
    parse_ideas_md, update_idea_status, append_learning_log,
    get_next_pending_idea, format_learning_log_entry
)
from utils.plateau import plateau_triggered, summarise_failures, ExperimentResult
from utils.experiment_runner import (
    implement_idea, validate_patch, run_experiment,
    git_commit, git_reset_hard, log_experiment, load_experiments,
    session_timeout_imminent, generate_experiment_id, get_file_hash,
    ExperimentLog
)
from utils.display import render_live_table, display_in_colab
from utils.branching import get_current_branch

# Reload checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)
checkpoint_dict = checkpoint.to_dict() if hasattr(checkpoint, 'to_dict') else vars(checkpoint)
comp_meta = checkpoint_dict.get('competition_meta', {})
metric_direction = comp_meta.get('metric_direction', 'higher_better')

SESSION_START = datetime.now()
experiment_results = []  # For plateau detection

print("=== Exploit Phase ===")
print(f"Starting from: {checkpoint.best_score:.6f}")
print(f"Baseline: {checkpoint.baseline_score:.6f}")
print("\nRunning experiment loop...\n")

# Get metric.py hash for validation
metric_path = REPO_DIR / 'metric.py'
metric_hash = get_file_hash(metric_path) if metric_path.exists() else None

try:
    while True:
        # Check for timeout
        if session_timeout_imminent(SESSION_START):
            print("\n⚠ Session timeout approaching. Saving state...")
            save_checkpoint(CHECKPOINT_PATH, checkpoint)
            break
        
        # Load ideas
        ideas = parse_ideas_md(IDEAS_PATH)
        next_idea = get_next_pending_idea(ideas)
        
        if next_idea is None:
            print("\n✓ All ideas exhausted!")
            break
        
        # Check plateau
        if plateau_triggered(
            checkpoint.plateau_window_scores,
            PLATEAU_WINDOW,
            PLATEAU_MIN_GAIN_PCT
        ):
            print("\n⚠ Plateau detected! Triggering re-research...")
            checkpoint = update_checkpoint_for_reresearch(checkpoint)
            save_checkpoint(CHECKPOINT_PATH, checkpoint)
            break  # Will continue in re-research cell
        
        # Mark idea as running
        update_idea_status(IDEAS_PATH, next_idea.title, 'running')
        
        print(f"\n--- Experiment {checkpoint.total_experiments + 1}: {next_idea.title} ---")
        start_time = time.time()
        
        # Implement the idea
        try:
            new_train_py = implement_idea(
                idea=next_idea,
                train_py_path=REPO_DIR / 'train.py',
                strategy_md_path=STRATEGY_PATH,
                client=client
            )
        except Exception as e:
            print(f"  Implementation error: {e}")
            update_idea_status(IDEAS_PATH, next_idea.title, 'crashed')
            continue
        
        # Validate patch
        is_valid, error = validate_patch(REPO_DIR, new_train_py, metric_hash)
        if not is_valid:
            print(f"  Validation failed: {error}")
            update_idea_status(IDEAS_PATH, next_idea.title, 'crashed')
            continue
        
        # Write new train.py
        with open(REPO_DIR / 'train.py', 'w') as f:
            f.write(new_train_py)
        
        # Run experiment
        score, error = run_experiment(REPO_DIR, SCALED_TIME_BUDGET)
        duration = time.time() - start_time
        
        # Determine outcome
        current_branch = get_current_branch(REPO_DIR)
        
        if score is None:
            # Crashed
            print(f"  CRASHED: {error[:100]}")
            git_reset_hard(REPO_DIR)
            status = 'crashed'
            update_idea_status(IDEAS_PATH, next_idea.title, 'crashed')
            
        elif (metric_direction == 'higher_better' and score > checkpoint.best_score) or \
             (metric_direction == 'lower_better' and score < checkpoint.best_score):
            # Improved!
            delta = score - checkpoint.best_score
            print(f"  IMPROVED: {checkpoint.best_score:.6f} → {score:.6f} ({delta:+.6f})")
            
            git_commit(REPO_DIR, f"IMPROVE [{current_branch}]: {next_idea.title} | {checkpoint.best_score:.4f} -> {score:.4f}")
            
            # Update learning log
            log_entry = format_learning_log_entry(
                checkpoint.total_experiments + 1,
                next_idea.title,
                'improved',
                score,
                delta
            )
            append_learning_log(STRATEGY_PATH, log_entry)
            
            status = 'improved'
            update_idea_status(IDEAS_PATH, next_idea.title, 'improved')
            checkpoint.best_score = score
            
        else:
            # No improvement
            print(f"  NO IMPROVEMENT: {score:.6f} (best: {checkpoint.best_score:.6f})")
            git_reset_hard(REPO_DIR)
            status = 'no_improvement'
            update_idea_status(IDEAS_PATH, next_idea.title, 'no_improvement')
        
        # Log experiment
        exp_log = ExperimentLog(
            id=generate_experiment_id(next_idea.title, datetime.now().isoformat()),
            idea_title=next_idea.title,
            idea_index=next_idea.index,
            branch=current_branch,
            status=status,
            score=score,
            previous_best=checkpoint.best_score if status != 'improved' else checkpoint.best_score - (score - checkpoint.best_score) if score else checkpoint.best_score,
            duration_seconds=duration,
            timestamp=datetime.now().isoformat(),
            train_py_hash=get_file_hash(REPO_DIR / 'train.py'),
            error_message=error,
        )
        log_experiment(DB_PATH, exp_log)
        
        # Track for plateau detection
        experiment_results.append(ExperimentResult(
            idea_title=next_idea.title,
            score=score,
            status=status,
            idea_index=next_idea.index
        ))
        
        # Update checkpoint
        checkpoint = update_checkpoint_after_experiment(
            checkpoint, score, status == 'improved', next_idea.index
        )
        save_checkpoint(CHECKPOINT_PATH, checkpoint)
        
        # Display live table
        clear_output(wait=True)
        experiments = load_experiments(DB_PATH)
        html = render_live_table(
            experiments, current_branch,
            checkpoint.best_score, checkpoint.baseline_score,
            metric_direction
        )
        display_in_colab(html)

except KeyboardInterrupt:
    print("\n\nLoop interrupted. Saving state...")
    save_checkpoint(CHECKPOINT_PATH, checkpoint)

print(f"\n=== Exploit Phase Complete ===")
print(f"Final best score: {checkpoint.best_score:.6f}")
print(f"Improvement from baseline: {checkpoint.best_score - checkpoint.baseline_score:+.6f}")

## Cell 7: Re-research Phase

This cell runs when a plateau is detected. It searches for new approaches and either:
- Adds new ideas to the current strategy
- Pivots to a new strategy (branches the codebase)

In [ ]:
from utils.reresearch import (
    reresearch_new_angle, reresearch_reread,
    handle_reresearch_result, should_attempt_reresearch
)
from utils.plateau import summarise_failures, ExperimentResult
from utils.branching import (
    archive_current_branch, start_new_branch,
    generate_branch_name, get_next_branch_version
)
from utils.strategy import archive_strategy
from utils.kaggle_api import get_template_for_problem_type
from utils.literature import load_cached_papers, format_papers_for_prompt
from utils.strategy import generate_ideas_md

# Reload checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)

if checkpoint.phase != 'reresearch':
    print("Not in re-research phase. Run the exploit loop first.")
else:
    print("=== Re-research Phase ===")
    print(f"Re-research attempt: {checkpoint.reresearch_attempts}")
    
    # Get failure summary
    experiments = load_experiments(DB_PATH)
    exp_results = [
        ExperimentResult(
            idea_title=e['idea_title'],
            score=e['score'],
            status=e['status'],
            idea_index=e['idea_index']
        ) for e in experiments
    ]
    failure_summary = summarise_failures(exp_results)
    
    checkpoint_dict = checkpoint.to_dict() if hasattr(checkpoint, 'to_dict') else vars(checkpoint)
    comp_meta = checkpoint_dict.get('competition_meta', {})
    
    # Attempt 1: New angle
    print("\nAttempt 1: Searching for new angles...")
    with open(STRATEGY_PATH, 'r') as f:
        strategy_md = f.read()
    
    result = reresearch_new_angle(
        strategy_md=strategy_md,
        failure_summary=failure_summary,
        search_history_path=LITERATURE_DIR / 'search_history.json',
        literature_cache_path=LITERATURE_DIR / 'papers.json',
        problem_type=checkpoint.problem_type,
        metric=comp_meta.get('metric', 'accuracy'),
        literature_depth=LITERATURE_DEPTH,
        client=client
    )
    
    print(f"Result: {result.outcome}")
    print(f"Reasoning: {result.reasoning}")
    
    if result.outcome == 'new_ideas':
        # Append new ideas and continue
        action = handle_reresearch_result(
            result, IDEAS_PATH, STRATEGY_PATH,
            LITERATURE_DIR / 'archived_strategies.md'
        )
        print(f"\n{action['message']}")
        checkpoint.phase = 'exploit'
        checkpoint.reresearch_attempts = 0
        save_checkpoint(CHECKPOINT_PATH, checkpoint)
        print("\nRun Cell 6 again to continue the exploit loop.")
        
    elif result.outcome == 'pivot':
        # Create new branch
        print("\n=== Pivot: Creating new strategy branch ===")
        
        version = get_next_branch_version(REPO_DIR)
        new_branch = generate_branch_name(result.pivot_strategy_name, version)
        old_branch = f"branch/strategy-v{version-1}" if version > 1 else "main"
        
        # Archive current branch
        archive_current_branch(REPO_DIR, old_branch)
        archive_strategy(STRATEGY_PATH, LITERATURE_DIR / 'archived_strategies.md')
        
        # Read baseline train.py
        template_name = get_template_for_problem_type(checkpoint.problem_type)
        template_path = Path(KAGGLERESEARCH_PATH) / 'templates' / template_name
        with open(template_path, 'r') as f:
            baseline_train = f.read()
        
        # Start new branch
        start_new_branch(REPO_DIR, new_branch, baseline_train)
        
        # Write new strategy
        with open(STRATEGY_PATH, 'w') as f:
            f.write(result.pivot_strategy_md)
        
        # Generate new IDEAS.md
        papers = load_cached_papers(LITERATURE_DIR / 'papers.json')
        papers_summary = format_papers_for_prompt(papers)
        
        new_ideas = generate_ideas_md(
            papers_summary=papers_summary,
            strategy_md=result.pivot_strategy_md,
            competition_meta={'name': comp_meta.get('name'), 'problem_type': checkpoint.problem_type},
            baseline_score=checkpoint.baseline_score,
            client=client
        )
        with open(IDEAS_PATH, 'w') as f:
            f.write(new_ideas)
        
        # Update checkpoint for branch comparison
        from utils.checkpoint import update_checkpoint_for_branch
        checkpoint = update_checkpoint_for_branch(
            checkpoint, new_branch, result.pivot_strategy_name
        )
        save_checkpoint(CHECKPOINT_PATH, checkpoint)
        
        print(f"\n✓ Created new branch: {new_branch}")
        print(f"✓ New strategy: {result.pivot_strategy_name}")
        print(f"\nRun Cell 6 to start experiments on the new branch.")
        print(f"After {BRANCH_COMPARE_N} experiments, run Cell 8 to compare branches.")
        
    else:  # no_new_ideas
        # Attempt 2: Re-read
        if should_attempt_reresearch(checkpoint.reresearch_attempts, max_attempts=2):
            print("\nAttempt 2: Re-reading papers with failure context...")
            result2 = reresearch_reread(
                LITERATURE_DIR / 'papers.json',
                strategy_md,
                failure_summary,
                client
            )
            
            if result2.outcome == 'new_ideas':
                action = handle_reresearch_result(
                    result2, IDEAS_PATH, STRATEGY_PATH,
                    LITERATURE_DIR / 'archived_strategies.md'
                )
                print(f"\n{action['message']}")
                checkpoint.phase = 'exploit'
                save_checkpoint(CHECKPOINT_PATH, checkpoint)
                print("\nRun Cell 6 again to continue.")
            else:
                print("\n⚠ Re-research exhausted. No new ideas found.")
                print("Strategy may have reached its ceiling.")
                checkpoint.phase = 'halted'
                save_checkpoint(CHECKPOINT_PATH, checkpoint)
        else:
            print("\n⚠ Maximum re-research attempts reached.")
            checkpoint.phase = 'halted'
            save_checkpoint(CHECKPOINT_PATH, checkpoint)

## Cell 8: Branch Comparison

After a pivot, this cell compares the old and new branches to determine which to continue.

In [ ]:
from utils.branching import compare_branches, promote_branch, switch_to_branch
from utils.checkpoint import update_checkpoint_after_branch_comparison
from utils.strategy import append_learning_log

# Reload checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)

if checkpoint.phase != 'branch_compare':
    print("Not in branch comparison phase.")
else:
    print("=== Branch Comparison ===")
    
    # Get branch scores
    branches = checkpoint.branches
    current_branch = checkpoint.current_branch
    new_best = checkpoint.best_score
    
    # Find the old branch
    old_branches = [b for b, info in branches.items() if info.get('status') == 'archived']
    if not old_branches:
        print("No archived branch found for comparison.")
    else:
        old_branch = old_branches[-1]  # Most recently archived
        old_best = branches[old_branch]['best_score']
        
        print(f"Old branch ({old_branch}): {old_best:.6f}")
        print(f"New branch ({current_branch}): {new_best:.6f}")
        
        # Compare
        checkpoint_dict = checkpoint.to_dict() if hasattr(checkpoint, 'to_dict') else vars(checkpoint)
        comp_meta = checkpoint_dict.get('competition_meta', {})
        metric_direction = comp_meta.get('metric_direction', 'higher_better')
        
        winner, loser = compare_branches(old_best, new_best, metric_direction)
        
        winner_branch = current_branch if winner == 'new' else old_branch
        loser_branch = old_branch if winner == 'new' else current_branch
        winner_score = new_best if winner == 'new' else old_best
        
        print(f"\n✓ Winner: {winner_branch} (score: {winner_score:.6f})")
        
        # Promote winner
        if winner == 'old':
            # Switch back to old branch
            switch_to_branch(REPO_DIR, old_branch)
            checkpoint.best_score = old_best
        
        # Update checkpoint
        checkpoint = update_checkpoint_after_branch_comparison(
            checkpoint, winner_branch, loser_branch
        )
        save_checkpoint(CHECKPOINT_PATH, checkpoint)
        
        # Add note to learning log
        note = f"Branch comparison: {winner_branch} won over {loser_branch} ({winner_score:.4f} vs {old_best if winner == 'new' else new_best:.4f})"
        append_learning_log(STRATEGY_PATH, note)
        
        print(f"\nContinuing with {winner_branch}. Run Cell 6 to continue experiments.")

## Cell 9: Summary

Generate a final summary of the research session.

In [ ]:
from utils.display import render_summary, display_in_colab
from utils.experiment_runner import load_experiments

# Reload checkpoint
checkpoint = load_checkpoint(CHECKPOINT_PATH)
checkpoint_dict = checkpoint.to_dict() if hasattr(checkpoint, 'to_dict') else vars(checkpoint)

print("=== KaggleResearch Summary ===")

summary_md = render_summary(
    log_path=DB_PATH,
    checkpoint=checkpoint_dict,
    baseline_score=checkpoint.baseline_score,
    strategy_md_path=STRATEGY_PATH
)

display_in_colab(summary_md, is_html=False)

# Save summary to file
with open(PROJECT_DIR / 'SUMMARY.md', 'w') as f:
    f.write(summary_md)
print(f"\n✓ Summary saved to {PROJECT_DIR / 'SUMMARY.md'}")

# Show submission location
submission_path = REPO_DIR / 'submissions' / 'submission.csv'
if submission_path.exists():
    print(f"\n📁 Submission file: {submission_path}")
    print("   Submit this file to Kaggle manually.")